In [ ]:
import torch

from utils import load_images, batch_numpy
from cnn import ConvNeuralNetwork

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = ConvNeuralNetwork(num_classes=17)
model.load_state_dict(torch.load("./model/model.pt"))
model.eval()

equal_images = load_images(
    s3_bucket_name="penman-lln",
    s3_bucket_path="data/token/="
)

equal_train_loader, equal_test_loader = batch_numpy(equal_images, batch_size=16, expected_output=14)

219it [00:26,  8.23it/s]


In [ ]:
with torch.no_grad():
    correct, total = 0, 0

    for images, labels in equal_train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the test images: {}%'.format(100 * correct / total))

Accuracy of the network on the test images: 86.36363636363636 %


In [ ]:
import onnxruntime as ort
import numpy as np

MODEL_PATH = "../numpy_cnn/model/penman_cnn.onnx"

session = ort.InferenceSession(MODEL_PATH)

label_decoding = {
    10: '(',
    11: ')',
    12: '+',
    13: '-',
    14: '=',
    15: 'fwd_slash',
    16: 'times'
}

output = session.run(None, {"X": images.astype(np.float32)})[0]
raw_values = list(np.argmax(output, axis=1))
str_values = [str(x) if x < 10 else label_decoding[x] for x in raw_values]

[1, 2, 3]
